# 05 — SAMap: how the integrations were built (READ-ONLY)

You asked to see this code but **not** recompute the embeddings. Nothing in this notebook refits
SAMap; it shows the exact inputs, normalization, and calls, and verifies the objects on disk match
what the code says it produced.

## The normalization question, settled

`prep_samap_input.py` puts **CP10K in `.X`** (library-size normalized to 1e4, **not** logged),
sourced from `layers['counts']`. Changed 2026-08-29; the raw-counts predecessor is preserved as
`prep_samap_input.py.raw_counts.bak`.

**Why CP10K and not raw counts** (verified in `sc_samap` 3.0.1 `sam/core.py`): SAM's
`preprocess_data` applies `sum_norm='cell_median'` -> `log2(x+1)` -> `min_expression=1` zeroing.
On CP10K input the sum-norm step is a **no-op** (every row already sums to 1e4), so exactly one log2
is applied; and because the `min_expression=1` cut lands *after* normalization, CP10K retains ~99.7%
of nonzero entries vs ~71% for raw counts at this library depth.

| `.X` contents | effective retention |
|---|---|
| raw counts + cell_median | 71.0% |
| **CP10K + cell_median** | **99.7%** |
| log1p(CP10K) | 91.4% — DO NOT USE (double-log) |

Two constraints the prep enforces, both load-bearing:
- **`.raw` is deleted.** SAM prefers `.raw.X` and discards `.X` when `.raw` exists — leaving it in
  silently swaps the scale.
- **Two-sided guard:** refuses to write unless row sums == 1e4 **and** `.X.max() > 20`. The max
  check is what catches an accidentally pre-logged matrix.

In [ ]:
# --- paths & helpers (edit ROOT if you move things) ---------------------------
import os, json, glob, pathlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

X    = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/xmens_final")
REV  = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/decidualization/revisions")
NET  = pathlib.Path("/n/netscratch/eddy_lab/Everyone/nhilgert")
GP   = REV/"crossspecies_radial_gp"
DELIV= X/"data/deliverables"
ISO  = X/"results/isodepth"
MIS  = X/"results/mi_stratified"
DATA = pathlib.Path("data")     # staged artifacts, see README
ORTH = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/human_ivd_repro/orthogroups/ensembl_human_mouse_components.tsv")

HS_DELIV = DELIV/"2026_08_28_Cevrim_Human_Only_Fib_Dec_Cells.h5ad"
MM_DELIV = DELIV/"2026_08_28_Cevrim_XMens_Only_Fib_Dec_Cells.h5ad"   # NOTE: "XMens", not "Mouse"

_checks = []
def verify(label, got, expected, tol=0.02, note=""):
    """Recompute vs recorded. Prints PASS/MISMATCH and accumulates a summary."""
    try:
        ok = (abs(float(got) - float(expected)) <= tol * max(1.0, abs(float(expected)))
              if expected is not None else None)
    except (TypeError, ValueError):
        ok = (got == expected)
    tag = "PASS " if ok else ("MISMATCH" if ok is False else "n/a  ")
    _checks.append((label, got, expected, tag))
    print(f"[{tag}] {label}: recomputed={got}  recorded={expected}" + (f"   ({note})" if note else ""))
    return ok

def check_summary():
    d = pd.DataFrame(_checks, columns=["claim","recomputed","recorded","status"])
    n_bad = int((d.status.str.strip()=="MISMATCH").sum())
    print(f"\n{len(d)} checks | {n_bad} MISMATCH")
    return d

def ortholog_1to1():
    o = pd.read_csv(ORTH, sep="\t")
    o = o[o.homology_type=="ortholog_one2one"]
    piv = o.pivot_table(index="component_id", columns="species", values="gene_symbol",
                        aggfunc=lambda s: s.iloc[0] if len(s)==1 else np.nan).dropna()
    return piv[["human","mouse"]].drop_duplicates()

print("data dir:", DATA.resolve(), "| files:", len(list(DATA.glob("*"))) if DATA.exists() else "MISSING")
print("env ok |", "deliverables:", HS_DELIV.exists(), MM_DELIV.exists(), "| isodepth dir:", ISO.exists())


## 5.1 The prep script, as installed

In [ ]:
prep = X/"workflow/scripts/prep_samap_input.py"
src = prep.read_text()
import re
keep = [l for l in src.splitlines()
        if re.search(r"TARGET_SUM|normalize|\.raw|rowsum|assert|max\(\)|counts|CP10K", l)]
print("\n".join(keep[:40]))
# NB: an earlier version of this check stripped spaces from src then searched for a
# spaced pattern, so it could never match. Test the semantics instead.
import re as _re
_ts = _re.search(r"TARGET_SUM\s*=\s*([0-9.e+]+)", src)
verify("prep defines TARGET_SUM = 1e4", float(_ts.group(1)) if _ts else None, 1e4, tol=0)
verify("prep scales .X by TARGET_SUM/rowsum",
       bool(_re.search(r"TARGET_SUM\s*/\s*rowsum", src)), True)
verify("prep deletes .raw", "del a.raw" in src or "a.raw = None" in src, True)
verify("raw-counts predecessor preserved", (X/"workflow/scripts/prep_samap_input.py.raw_counts.bak").exists(), True)

## 5.2 The SAMap call — stock/vanilla, no harmony

In [ ]:
run = X/"workflow/scripts/run_samap.py"
s = run.read_text()
for l in s.splitlines():
    if re.search(r"SAMAP\(|\.run\(|import samap|f_maps|NUMITERS|harmony", l):
        print("   ", l.strip())
verify("stock SAMap (no harmony in the pipeline arm)", "harmony" not in s.lower(), True,
       note="harmony variant exists separately as the robustness supplement")

## 5.3 Verify the objects on disk

Note: the **exported joint objects** carry SAM's *post-processing* scale (row sums ~5,400, max ~10.2)
— that is `log2` applied after `cell_median`, i.e. what SAM produced internally. The **CP10K is the
input**, not what you read back out of the joint object. If you need CP10K or counts, go to
`layers['counts']` in the deliverables.

In [ ]:
import scanpy as sc
for f in ("2026_08_31_samap_vanilla_fibdec_joint.h5ad",):
    p_ = NET/f
    if not p_.exists(): print("missing", p_); continue
    a = sc.read_h5ad(p_, backed="r")
    Xs = a[:400].to_memory().X
    Xd = Xs.toarray() if hasattr(Xs,"toarray") else np.asarray(Xs)
    rs = Xd.sum(1)
    print(f"{f}\n  shape={a.shape}  rowsum med={np.median(rs):.0f}  max={Xd.max():.2f}  layers={list(a.layers)}")
    verify("joint object .X is SAM-processed (not CP10K)", bool(abs(np.median(rs)-1e4) > 1000), True,
           note="row sums ~5.4k, max ~10.2 => post cell_median + log2")
    if "export_provenance" in a.uns:
        print("  provenance:", str(a.uns["export_provenance"])[:300])
    del a

## 5.4 Coarse mapping scores across the 12 individual-sample pairings

The specificity (diagonal / off-diagonal) peaks at **1dpi_late–2dpi** in *both* human references —
that reproducibility is what makes it credible.

In [ ]:
rows = []
for f in sorted(glob.glob(str(DATA/"mapping_scores_*_hmmer.csv"))):
    run_ = os.path.basename(f).replace("mapping_scores_","").replace("_hmmer.csv","")
    M = pd.read_csv(f, index_col=0)
    g = lambda r_, c_: float(M.loc[r_, c_]) if (r_ in M.index and c_ in M.columns) else np.nan
    rows.append(dict(run=run_,
        dec_dec=g("hs_decidual_cell","mm_decidual_cell"),
        fib_fib=g("hs_endometrial_fibroblast","mm_endometrial_fibroblast"),
        off1=g("hs_decidual_cell","mm_endometrial_fibroblast"),
        off2=g("hs_endometrial_fibroblast","mm_decidual_cell")))
if rows:
    T = pd.DataFrame(rows)
    T["diag_over_off"] = ((T.dec_dec+T.fib_fib)/2) / ((T.off1+T.off2)/2)
    order = ["uninduced","1dpi_early","1dpi_late","2dpi","3dpi","4dpi"]
    T["tp"] = T.run.str.split("_x_").str[1]; T["phase"] = T.run.str.split("_x_").str[0]
    T = T.sort_values(["phase","tp"], key=lambda s: s.map({v:i for i,v in enumerate(order)}) if s.name=="tp" else s)
    print(T[["phase","tp","dec_dec","fib_fib","off1","off2","diag_over_off"]].round(3).to_string(index=False))
    best = T.loc[T.diag_over_off.idxmax()]
    verify("specificity peaks at 1dpi_late or 2dpi", best.tp in ("1dpi_late","2dpi"), True,
           note=f"peak at {best.phase} x {best.tp}, ratio {best.diag_over_off:.2f}")
else:
    print("mapping_scores_*.csv not found - they are in the genepairs job workdir")

## Summary

In [ ]:
check_summary()